# opencompass inference

In [1]:
# =========================
# CELL 1: install & sanity check
# =========================

import subprocess
import sys

print("Python executable for this kernel:", sys.executable)

# Quick sanity check: run OpenCompass via THIS Python, not via PATH
result0 = subprocess.run(
    [sys.executable, "pwd"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(result0, flush=True)

result = subprocess.run(
    [sys.executable, "-m", "opencompass", "--help"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("\nFirst 15 lines of `python -m opencompass --help`:\n")
print("\n".join(result.stdout.splitlines()[:15]))


Python executable for this kernel: /Users/aneeshkalla/anaconda3/envs/restruct/bin/python
CompletedProcess(args=['/Users/aneeshkalla/anaconda3/envs/restruct/bin/python', 'pwd'], returncode=2, stdout="/Users/aneeshkalla/anaconda3/envs/restruct/bin/python: can't open file '/Users/aneeshkalla/Desktop/Restruct/backend_code/notebooks/pwd': [Errno 2] No such file or directory\n")

First 15 lines of `python -m opencompass --help`:

/Users/aneeshkalla/anaconda3/envs/restruct/bin/python: No module named opencompass.__main__; 'opencompass' is a package and cannot be directly executed


In [2]:
import os
import sys
import subprocess
from pathlib import Path

def run_restruct_opencompass_infer(
    work_dir: str = "outputs",
    extra_args: list[str] | None = None,
):
    python_exe = Path(sys.executable)
    oc_exe = python_exe.with_name("opencompass")

    print("Kernel Python:", python_exe)
    print("Using opencompass executable:", oc_exe)

    if not oc_exe.exists():
        raise FileNotFoundError(f"{oc_exe} does not exist. Is OpenCompass installed in this env?")

    # Ensure notebook dir (with restruct_oc_model.py) is on PYTHONPATH for child processes
    env = os.environ.copy()
    notebooks_dir = str(Path.cwd())
    env["PYTHONPATH"] = notebooks_dir + os.pathsep + env.get("PYTHONPATH", "")

    print("PYTHONPATH for subprocess:", env["PYTHONPATH"])

    cmd = [
        str(oc_exe),
        "restruct_opencompass_config.py",
        "-w",
        work_dir,
        "-m",
        "infer",              # <<< ONLY RUN INFERENCE
    ]
    if extra_args:
        cmd.extend(extra_args)

    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, env=env)

# Call it:
run_restruct_opencompass_infer()


Kernel Python: /Users/aneeshkalla/anaconda3/envs/restruct/bin/python
Using opencompass executable: /Users/aneeshkalla/anaconda3/envs/restruct/bin/opencompass
PYTHONPATH for subprocess: /Users/aneeshkalla/Desktop/Restruct/backend_code/notebooks:
Running: /Users/aneeshkalla/anaconda3/envs/restruct/bin/opencompass restruct_opencompass_config.py -w outputs -m infer


/Users/aneeshkalla/anaconda3/envs/restruct/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Traceback (most recent call last):
  File "/Users/aneeshkalla/anaconda3/envs/restruct/bin/opencompass", line 7, in <module>
    sys.exit(main())
  File "/Users/aneeshkalla/Desktop/Restruct/third_party/opencompass/opencompass/cli/main.py", line 263, in main
    cfg = get_config_from_arg(args)
  File "/Users/aneeshkalla/Desktop/Restruct/third_party/opencompass/opencompass/utils/run.py", line 98, in get_config_from_arg
    config = Config.fromfile(args.config, format_python_code=False)
  File "/Users/aneeshkalla/anaconda3/envs/restruct/lib/python3.10/site-packages/mmengine/config/config.py", line 494, in fromfile
    raise e
  File "/Use

CalledProcessError: Command '['/Users/aneeshkalla/anaconda3/envs/restruct/bin/opencompass', 'restruct_opencompass_config.py', '-w', 'outputs', '-m', 'infer']' returned non-zero exit status 1.

In [29]:
import pandas as pd
from pathlib import Path

def update_predictions_df(run_dir: str, df: pd.DataFrame | None = None) -> pd.DataFrame:
    run_path = Path(run_dir)
    preds_dir = run_path / "predictions"
    if not preds_dir.exists():
        raise FileNotFoundError(f"No predictions directory at {preds_dir}")

    # Start from existing CSV if not provided
    if df is None:
        csv_path = run_path.parent.parent / "predictions.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path, index_col=0)
        else:
            df = pd.DataFrame()

    # Traverse model/dataset files
    for model_dir in preds_dir.iterdir():
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        for json_file in model_dir.glob("*.json"):
            dataset_name = json_file.stem  # e.g., demo_gsm8k
            if model_name not in df.index:
                df.loc[model_name, :] = pd.NA
            if dataset_name not in df.columns:
                df[dataset_name] = pd.NA
            df.at[model_name, dataset_name] = str(json_file.resolve())

    return df

# Example usage:
base_run = "/Users/aneeshkalla/Desktop/Restruct/backend_code/notebooks/outputs/20251129_042703"
csv_out = Path(base_run).parent.parent / "predictions.csv"

df = update_predictions_df(base_run)
df.to_csv(csv_out)
df


,demo_gsm8k
restruct_google_gemini_2.0_flash_lite,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_anthropic_claude_opus_4_1,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_anthropic_claude_sonnet_4_5,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_google_gemini_2.5_flash_lite,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_openai_gpt_5,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_openai_gpt_5_mini,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_google_gemini_2.0_flash,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_google_gemini_2.5_pro,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_anthropic_claude_haiku_4_5,/Users/aneeshkalla/Desktop/Restruct/backend_co...
restruct_openai_gpt_5_nano,/Users/aneeshkalla/Desktop/Restruct/backend_co...


# grading

In [ ]:
#GRADER V0 (OLD VERSION DONT USE ANYMORE)

'''
import sys, re, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

# Make backend_code importable
repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend_code import inference, models_config  # noqa: E402

OPENAI_CFG = models_config.MODELS['openai']
SCORER_MODEL = {
    'vendor': 'openai',
    'model_name': 'gpt-5-mini',
    'api_key': OPENAI_CFG['api_key'],
    'config': OPENAI_CFG['models']['gpt-5-mini'],
}

MAX_WORKERS = 32  # stay well under 400 rate limit

def grader0(model_output: str, gold_output: str) -> float:
    system_prompt = (
        'You are a strict grader. Score the model answer against the reference '
        'from 0 to 1, where 1 means fully correct and 0 means completely wrong. '
        'Return ONLY JSON like {"score": <float between 0 and 1>}.'
    )
    user_prompt = (
        f"Reference answer:{gold_output}"
        f"Model answer:{model_output}"
        'Respond with JSON only.'
    )
    conversation = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    resp = inference.infer(SCORER_MODEL, conversation)
    text = resp.get('text', '')
    match = re.search(r'([0-1](?:\.\d+)?|\.\d+)', text)
    if not match:
        raise ValueError(f'Could not parse score from response: {text}')
    score = float(match.group(1))
    return max(0.0, min(1.0, score))


def grade_file(json_path: Path) -> None:
    data = json.loads(json_path.read_text())
    items = [(k, v) for k, v in data.items() if 'grader0' not in v]
    if not items:
        return

    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(grader0, entry.get('prediction', ''), entry.get('gold', '')): key for key, entry in items}
        with tqdm(total=len(futures), desc=json_path.name, leave=False) as pbar:
            for fut in as_completed(futures):
                key = futures[fut]
                results[key] = fut.result()
                pbar.update(1)

    for key, score in results.items():
        data[key]['grader0'] = score

    json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2))


def grade_from_csv(csv_path: Path) -> None:
    df = pd.read_csv(csv_path, index_col=0)
    json_paths = []
    for _, row in df.iterrows():
        for path in row.values:
            if isinstance(path, str) and path.strip():
                jp = Path(path)
                if jp.exists():
                    json_paths.append(jp)
    with tqdm(total=len(json_paths), desc='Files') as files_bar:
        for jp in json_paths:
            grade_file(jp)
            files_bar.update(1)


csv_path = Path('predictions.csv')
grade_from_csv(csv_path)
print('Grading complete')
'''

Files: 100%|██████████| 11/11 [00:00<00:00, 288.52it/s]

Grading complete


In [ ]:
#GRADER V1

import sys, re, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

# Make backend_code importable
repo_root = Path.cwd().parents[1]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from backend_code import inference, models_config  # noqa: E402

OPENAI_CFG = models_config.MODELS['openai']
SCORER_MODEL = {
    'vendor': 'openai',
    'model_name': 'gpt-5-mini',
    'api_key': OPENAI_CFG['api_key'],
    'config': OPENAI_CFG['models']['gpt-5-mini'],
}

MAX_WORKERS = 32  # stay well under 400 rate limit

def grader1(prompt_text:str, model_output: str, gold_output: str) -> float:
    system_prompt = (
'''You are grading an AI assistant's answer to a user's request.

You will be given:
- The user's request (the input or question)
- The reference or ideal answer (what a very good assistant would say)
- The assistant's actual answer

Your job is to:
1. Judge how well the assistant's answer fulfills the user's request.
2. Compare it to the reference answer:
   - Is it correct and accurate?
   - Does it follow the instructions and format?
   - Is it clear and helpful?
3. Give a continuous score between 0.0 and 1.0, where:
   - 1.0 = completely correct or excellent; fully satisfies the request.
   - 0.75 = mostly correct, only minor issues or omissions.
   - 0.5 = partially correct; some important elements are right, but there are clear mistakes or missing pieces.
   - 0.25 = minimally helpful; a few relevant points, but largely wrong, off-topic, or low quality.
   - 0.0 = completely incorrect, off-topic, or nonsensical.

You may use any real value in [0.0, 1.0], not just these examples.

First, briefly explain your reasoning in 1–3 sentences.
Then, on the LAST LINE, output a single JSON object in this exact format:

{"score": <number between 0.0 and 1.0>}'''
    )
    user_prompt = (
        f'''
        Now grade the following: 

        User Request:
        {prompt_text}

        Reference / ideal answer:
        {gold_output}

        Assistant's answer:
        {model_output}
    '''
    )
    conversation = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    resp = inference.infer(SCORER_MODEL, conversation)
    text = resp.get('text', '')

    # Find the last {...} JSON block in the entire text
    start = text.rfind('{')
    end = text.rfind('}')

    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Could not find JSON object in judge response: {text}")

    json_str = text[start:end+1]

    try:
        data = json.loads(json_str)
        score = float(data.get("score", 0.0))
    except Exception as e:
        raise ValueError(f"Invalid JSON in judge output: {json_str}") from e

    # Clamp to [0, 1]
    score = max(0.0, min(1.0, score))

    return score

def conversation_to_text(origin_prompt):
    parts = []
    for msg in origin_prompt:
        role = msg.get('role', '').upper()
        content = msg.get('prompt', '') or msg.get('content', '')
        parts.append(f"{role}: {content}")
    return "".join(parts)


def grade_file(json_path: Path) -> None:
    data = json.loads(json_path.read_text())
    items = [(k, v) for k, v in data.items() if 'grader1' not in v]
    if not items:
        return

    results = {}
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    
        futures = {ex.submit(grader1, conversation_to_text(entry.get('origin_prompt', [])), entry.get('prediction', ''), entry.get('gold', '')): key for key, entry in items}
        with tqdm(total=len(futures), desc=json_path.name, leave=False) as pbar:
            for fut in as_completed(futures):
                key = futures[fut]
                results[key] = fut.result()
                pbar.update(1)

    for key, score in results.items():
        data[key]['grader1'] = score

    json_path.write_text(json.dumps(data, ensure_ascii=False, indent=2))


def grade_from_csv(csv_path: Path) -> None:
    df = pd.read_csv(csv_path, index_col=0)
    json_paths = []
    for _, row in df.iterrows():
        for path in row.values:
            if isinstance(path, str) and path.strip():
                jp = Path(path)
                if jp.exists():
                    json_paths.append(jp)
    with tqdm(total=len(json_paths), desc='Files') as files_bar:
        for jp in json_paths:
            grade_file(jp)
            files_bar.update(1)


csv_path = Path('predictions.csv')
grade_from_csv(csv_path)
print('Grading complete')


In [5]:
csv_path = Path('predictions.csv')
grade_from_csv(csv_path)
print('Grading complete')

Files: 100%|██████████| 11/11 [00:00<00:00, 398.12it/s]

Grading complete


# vector stuff

In [ ]:
import json
import pandas as pd
from pathlib import Path

# Load predictions.csv
csv_path = Path('predictions.csv')
df_paths = pd.read_csv(csv_path, index_col=0)

# Helper to collapse origin_prompt list into a single text string

def conversation_to_text(origin_prompt):
    parts = []
    for msg in origin_prompt:
        role = msg.get('role', '').upper()
        content = msg.get('prompt', '') or msg.get('content', '')
        parts.append(f"{role}: {content}")
    return "".join(parts)

# Build dataframe: rows = prompt text, columns = models, values = grader0 score
rows = {}
for model_name, row in df_paths.iterrows():
    for dataset, path in row.items():
        if not isinstance(path, str) or not path.strip():
            continue
        data = json.loads(Path(path).read_text())
        for entry in data.values():
            prompt_text = conversation_to_text(entry.get('origin_prompt', []))
            score = entry.get('grader0')
            if score is None:
                continue
            if prompt_text not in rows:
                rows[prompt_text] = {}
            rows[prompt_text][model_name] = score

prompt_model_scores = pd.DataFrame.from_dict(rows, orient='index')
prompt_model_scores.index.name = 'prompt'
prompt_model_scores



In [ ]:
# Embed each prompt and add as a new column
from tqdm import tqdm
from backend_code.embedding import embed_prompt

vectors = []
for prompt in tqdm(prompt_model_scores.index):
    vectors.append(embed_prompt(prompt))

prompt_model_scores['vector'] = vectors

# Save to CSV
a = prompt_model_scores.copy()
a['vector'] = a['vector'].apply(json.dumps)
a.to_csv('vectors_v0.csv')

prompt_model_scores

In [15]:
len(prompt_model_scores.iloc[0]["vector"])

1536